<a href="https://colab.research.google.com/github/taruchit/Using-RAG-for-Planning-stay-in-Rajasthan/blob/pdf_parsing/pdf_plumber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 72.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 106.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 108.4 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pdfplumber]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [5]:
!pip install -qU langchain-openai

In [22]:
!pip install -qU pypdfium2 pillow langchain-openai

In [23]:
import os
import pdfplumber
from langchain_openai import ChatOpenAI
import base64
import io
import pypdfium2 as pdfium
from langchain_core.messages import HumanMessage

In [18]:
from google.colab import userdata

# Securely retrieve the OpenAI API key from Colab Secrets
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Fallback to os.environ if not found in Colab Secrets
    openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("Please set your 'OPENAI_API_KEY' in the Colab Secrets (the key icon on the left sidebar).")

In [20]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)

In [21]:
def process_table_with_llm(table):
  cleaned_table = [row for row in table if row and any(cell for cell in row)]
  if not cleaned_table:
    return ""

  prompt = f"""Convert the following table enclosed in backticks into readable text:
```{cleaned_table}```
"""
  response = llm.invoke(prompt)
  return response.content


all_output = []

with pdfplumber.open("/content/Tariff.pdf") as pdf:
  print(f"Total pages: {len(pdf.pages)}")

  for idx, page in enumerate(pdf.pages, start=1):
    text = page.extract_text() or ""
    tables = page.extract_tables() or []
    print(f"Page {idx}: {len(text)} chars of text, {len(tables)} tables")

    page_table_summaries = []
    for table in tables:
      summary = process_table_with_llm(table)
      if summary:
        page_table_summaries.append(summary)

    combined_tables = "\n\n".join(page_table_summaries)
    page_block = (
        f"=== Page {idx} ===\n"
        f"{text}\n\n"
        f"*** Table summary ***\n"
        f"{combined_tables if combined_tables else 'No tables detected.'}\n"
    )
    all_output.append(page_block)

final_text = "\n\n".join(all_output)
print(final_text)

with open("tariff_summary.txt", "w", encoding="utf-8") as output_file:
  output_file.write(final_text)

print("Saved output to tariff_summary.txt")

Total pages: 15
Page 1: 0 chars of text, 0 tables
Page 2: 0 chars of text, 0 tables
Page 3: 0 chars of text, 0 tables
Page 4: 0 chars of text, 0 tables
Page 5: 0 chars of text, 0 tables
Page 6: 0 chars of text, 0 tables
Page 7: 0 chars of text, 0 tables
Page 8: 0 chars of text, 0 tables
Page 9: 0 chars of text, 0 tables
Page 10: 0 chars of text, 0 tables
Page 11: 0 chars of text, 0 tables
Page 12: 0 chars of text, 0 tables
Page 13: 0 chars of text, 0 tables
Page 14: 0 chars of text, 0 tables
Page 15: 0 chars of text, 0 tables
=== Page 1 ===


*** Table summary ***
No tables detected.


=== Page 2 ===


*** Table summary ***
No tables detected.


=== Page 3 ===


*** Table summary ***
No tables detected.


=== Page 4 ===


*** Table summary ***
No tables detected.


=== Page 5 ===


*** Table summary ***
No tables detected.


=== Page 6 ===


*** Table summary ***
No tables detected.


=== Page 7 ===


*** Table summary ***
No tables detected.


=== Page 8 ===


*** Table summary ***
No

In [24]:
def pil_to_base64_png(image) -> str:
  buf = io.BytesIO()
  image.save(buf, format="PNG")
  return base64.b64encode(buf.getvalue()).decode("utf-8")


pdf = pdfium.PdfDocument("/content/Tariff.pdf")
print(f"Total pages: {len(pdf)}")

all_output = []

for idx, page in enumerate(pdf, start=1):
  # Render at ~200 DPI (scale ~2.78 from 72 DPI) for crisp small tariff text
  bitmap = page.render(scale=2.78)
  pil_image = bitmap.to_pil()
  b64_img = pil_to_base64_png(pil_image)

  message = HumanMessage(
      content=[
          {
              "type": "text",
              "text": (
                  f"Extract all text and convert any tables on Page {idx} into "
                  "clear, readable text while preserving rates, codes, and rows."
              ),
          },
          {
              "type": "image_url",
              "image_url": {"url": f"data:image/png;base64,{b64_img}"},
          },
      ]
  )

  response = llm.invoke([message])
  page_block = f"=== Page {idx} ===\n{response.content}\n"
  all_output.append(page_block)
  print(f"Processed Page {idx}/{len(pdf)}")

final_text = "\n\n".join(all_output)
with open("tariff_summary.txt", "w", encoding="utf-8") as f:
  f.write(final_text)

print("Saved output to tariff_summary.txt")

Total pages: 15
Processed Page 1/15
Processed Page 2/15
Processed Page 3/15
Processed Page 4/15
Processed Page 5/15
Processed Page 6/15
Processed Page 7/15
Processed Page 8/15
Processed Page 9/15
Processed Page 10/15
Processed Page 11/15
Processed Page 12/15
Processed Page 13/15
Processed Page 14/15
Processed Page 15/15
Saved output to tariff_summary.txt


In [26]:
from google.colab import drive
drive.mount('/content/drive')

# Example: Saving a CSV or text file to your Drive
file_path = '/content/drive/MyDrive/tariff_summary.txt'

# Replace this with how you normally save your file (e.g., df.to_csv, open(), etc.)
with open("tariff_summary.txt", "w", encoding="utf-8") as f:
  f.write(final_text)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
